## 1. Название лабораторной работы

# Лабораторная работа №1. Сбор, хранение и агрегация данных о землетрясениях

**Выполнили:** Марюшкина Софья и Чуева Арина, поток 1.3.

**Источник:** [USGS Earthquake Catalog](https://earthquake.usgs.gov/fdsnws/event/1/).


## 2. Цель и задачи лабораторной работы

**Цель:** собрать данные о землетрясениях с веб-ресурса USGS, организовать их хранение и пополнение в SQLite, выполнить агрегирование средствами pandas и SQLAlchemy.

**Задачи:**

1. Получить события через API USGS с помощью Scrapy и преобразовать GeoJSON в строки таблицы.
2. Создать таблицу SQLite и сохранить полученные данные.
3. Обеспечить добавление новых событий без дубликатов при повторном сборе и обновление уточнённых записей.
4. Выполнить не менее пяти разных запросов для агрегации с использованием pandas и SQLAlchemy.
5. Сопоставить результаты двух способов обработки и сформулировать выводы.


## 3. Описание входных данных

USGS предоставляет каталог событий через HTTP API. Запрос отправляется на `https://earthquake.usgs.gov/fdsnws/event/1/query`, ответ имеет формат GeoJSON. Список `features` содержит события: идентификатор, свойства и геометрию с координатами в порядке «долгота, широта, глубина».

Выборка: землетрясения магнитудой **от 2,5** за **1–14 января 2025 года**. Все даты и время приведены к UTC. Сбор разделён на два интервала: `[2025-01-01, 2025-01-08)` и `[2025-01-08, 2025-01-15)`. Начало включено, конец не включён.

Одна строка базы соответствует одному событию. Собираемые поля:

| Поле | Описание |
|---|---|
| `event_id` | Уникальный ID события USGS, первичный ключ |
| `occurred_at` | Дата и время события в UTC |
| `updated_ms` | Время последнего обновления записи в USGS, миллисекунды Unix |
| `magnitude` | Магнитуда |
| `depth_km` | Глубина, км |
| `latitude`, `longitude` | Координаты |
| `place` | Описание места |
| `network` | Код сети, предоставившей запись |
| `status` | Статус проверки события |
| `tsunami` | Флаг tsunami из каталога USGS |
| `significance` | Показатель значимости события USGS |
| `url` | Ссылка на событие |

Пропуски необязательных полей сохраняются как NULL. Записи без ID, магнитуды, времени события, времени обновления или координат пропускаются. Каталог может уточняться после первоначальной публикации, поэтому сохраняется `updated_ms`. Код сети обозначает источник записи и не является названием страны. Флаг `tsunami` сам по себе не доказывает возникновение цунами.


## 4. Описание этапов выполнения работы

Код проекта распределён между `earthquake_project/spiders/scraper.py` (сбор), `earthquake_project/models.py` (хранение) и `analysis.py` (агрегации). Ниже представлены код этапов и результаты их выполнения. Блокнот использует эти модули из корня проекта; зависимости указаны в `requirements.txt`.

### 4.1. Подготовка окружения

Для каждого выполнения опыта создаётся новая SQLite-база в `data/report_runs/`. Это позволяет наблюдать первую загрузку и догрузку, сохраняя ранее созданные базы. Версии библиотек и дата выполнения фиксируются в отчёте.


In [1]:
import json
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from sqlalchemy import case, func, inspect, select

from earthquake_project.models import PROJECT_DIR, Earthquake, create_database
from analysis import compare_results

executed_at = datetime.now(timezone.utc)
report_db = PROJECT_DIR / "data" / "report_runs" / f"report_{executed_at:%Y%m%d_%H%M%S_%f}.sqlite"
print("Дата выполнения (UTC):", executed_at.strftime("%Y-%m-%d %H:%M:%S"))
display(pd.DataFrame({
    "Компонент": ["Python", "Scrapy", "SQLAlchemy", "pandas"],
    "Версия": [sys.version.split()[0], version("Scrapy"), version("SQLAlchemy"), version("pandas")],
}))


Дата выполнения (UTC): 2026-10-05 18:43:50


,Компонент,Версия
0,Python,3.12.0
1,Scrapy,2.19.0
2,SQLAlchemy,2.1.2
3,pandas,2.3.3


### 4.2. Создание базы данных и сохранение записей

SQLAlchemy описывает таблицу `earthquakes`. Первичный ключ `event_id` соответствует ID события USGS. Функция `save_earthquakes` проверяет наличие события: добавляет новое либо обновляет существующее, если источник прислал более поздний `updated_ms`. Старые версии не заменяют новые.

Код модели и сохранения из `earthquake_project/models.py`:


```python
from pathlib import Path

from sqlalchemy import URL, Float, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

PROJECT_DIR = Path(__file__).resolve().parent.parent
DB_PATH = PROJECT_DIR / "data" / "earthquakes.sqlite"


class Base(DeclarativeBase):
    pass


class Earthquake(Base):
    __tablename__ = "earthquakes"

    event_id: Mapped[str] = mapped_column(String, primary_key=True)
    occurred_at: Mapped[str] = mapped_column(String)
    updated_ms: Mapped[int] = mapped_column(Integer)
    magnitude: Mapped[float] = mapped_column(Float)
    depth_km: Mapped[float | None] = mapped_column(Float)
    latitude: Mapped[float | None] = mapped_column(Float)
    longitude: Mapped[float | None] = mapped_column(Float)
    place: Mapped[str | None] = mapped_column(String)
    network: Mapped[str | None] = mapped_column(String)
    status: Mapped[str | None] = mapped_column(String)
    tsunami: Mapped[int | None] = mapped_column(Integer)
    significance: Mapped[int | None] = mapped_column(Integer)
    url: Mapped[str | None] = mapped_column(String)


def create_database(db_path=DB_PATH):
    path = Path(db_path).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    engine = create_engine(URL.create("sqlite", database=str(path)))
    Base.metadata.create_all(engine)
    return engine


def save_earthquakes(rows, engine):
    counts = {"inserted": 0, "updated": 0, "unchanged": 0}
    with Session(engine) as session:
        for row in rows:
            existing = session.get(Earthquake, row["event_id"])
            if existing is None:
                session.add(Earthquake(**row))
                counts["inserted"] += 1
            elif row["updated_ms"] > existing.updated_ms:
                for field, value in row.items():
                    setattr(existing, field, value)
                counts["updated"] += 1
            else:
                counts["unchanged"] += 1
        session.commit()
    return counts
```


In [2]:
engine = create_database(report_db)
with engine.connect() as connection:
    initial_count = connection.scalar(select(func.count()).select_from(Earthquake))
assert initial_count == 0
print("Строк в новой базе:", initial_count)
display(pd.DataFrame([
    {"Поле": field["name"], "Тип SQLite": str(field["type"]),
     "Первичный ключ": field["primary_key"] == 1}
    for field in inspect(engine).get_columns("earthquakes")
]))


Строк в новой базе: 0


,Поле,Тип SQLite,Первичный ключ
0,event_id,VARCHAR,True
1,occurred_at,VARCHAR,False
2,updated_ms,INTEGER,False
3,magnitude,FLOAT,False
4,depth_km,FLOAT,False
5,latitude,FLOAT,False
6,longitude,FLOAT,False
7,place,VARCHAR,False
8,network,VARCHAR,False
9,status,VARCHAR,False


### 4.3. Сбор данных с USGS

Scrapy отправляет запросы с датами, порогом магнитуды и параметрами `limit` и `offset`. Следующая страница запрашивается, если текущая заполнена целиком. Из каждого объекта GeoJSON извлекаются свойства события и координаты; подготовленные строки передаются в `save_earthquakes`.

В API USGS граница `endtime` включена, поэтому для интервала с исключённой конечной датой вычитается одна миллисекунда.

Код сборщика из `earthquake_project/spiders/scraper.py`:


```python
from datetime import date, datetime, time, timedelta, timezone
from urllib.parse import urlencode

import scrapy

from earthquake_project.models import DB_PATH, create_database, save_earthquakes


class EarthquakesSpider(scrapy.Spider):
    name = "earthquakes"
    allowed_domains = ["earthquake.usgs.gov"]
    handle_httpstatus_list = [204]
    api_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

    def __init__(self, start="2025-01-01", end="2025-01-15",
                 min_magnitude="2.5", page_size="1000", db_path=None, **kwargs):
        super().__init__(**kwargs)
        start_date, end_date = date.fromisoformat(start), date.fromisoformat(end)
        if start_date >= end_date:
            raise ValueError("Дата end должна быть позже start.")
        self.page_size = int(page_size)
        if not 1 <= self.page_size <= 20_000:
            raise ValueError("page_size должен быть от 1 до 20000.")
        # USGS включает endtime; вычитание миллисекунды задаёт период [start, end).
        end_time = datetime.combine(end_date, time.min, tzinfo=timezone.utc)
        end_time -= timedelta(milliseconds=1)
        self.params = {
            "format": "geojson",
            "eventtype": "earthquake",
            "starttime": start_date.isoformat(),
            "endtime": end_time.isoformat(timespec="milliseconds"),
            "minmagnitude": float(min_magnitude),
            "orderby": "time-asc",
            "limit": self.page_size,
        }
        self.engine = create_database(db_path or DB_PATH)

    def page_request(self, offset):
        params = {**self.params, "offset": offset}
        return scrapy.Request(
            f"{self.api_url}?{urlencode(params)}",
            callback=self.parse,
            cb_kwargs={"offset": offset},
        )

    async def start(self):
        yield self.page_request(1)

    def parse(self, response, offset=1):
        if response.status == 204:
            return
        features = response.json()["features"]
        rows = []
        for feature in features:
            props = feature.get("properties") or {}
            coords = (feature.get("geometry") or {}).get("coordinates") or []
            if (not feature.get("id") or props.get("mag") is None
                    or props.get("time") is None or props.get("updated") is None
                    or len(coords) < 2):
                self.crawler.stats.inc_value("events/skipped")
                continue
            rows.append({
                "event_id": feature["id"],
                "occurred_at": datetime.fromtimestamp(
                    props["time"] / 1000, tz=timezone.utc
                ).isoformat(timespec="milliseconds"),
                "updated_ms": int(props["updated"]),
                "magnitude": float(props["mag"]),
                "depth_km": coords[2] if len(coords) > 2 else None,
                "latitude": coords[1],
                "longitude": coords[0],
                "place": props.get("place"),
                "network": props.get("net"),
                "status": props.get("status"),
                "tsunami": int(props.get("tsunami") or 0),
                "significance": props.get("sig"),
                "url": props.get("url"),
            })

        counts = save_earthquakes(rows, self.engine)
        self.crawler.stats.inc_value("events/received", len(features))
        for name, count in counts.items():
            self.crawler.stats.inc_value(f"events/{name}", count)
        self.logger.info(
            "Получено: %s; добавлено: %s; обновлено: %s; без изменений: %s",
            len(features), counts["inserted"], counts["updated"], counts["unchanged"],
        )
        if len(features) == self.page_size:
            yield self.page_request(offset + len(features))

    def closed(self, reason):
        self.engine.dispose()
```


В блокноте каждый сбор запускается в отдельном процессе Python, чтобы повторные обращения к Scrapy не конфликтовали с циклом событий Jupyter. Процесс возвращает статистику загрузки. Ошибки HTTP или обработки прерывают опыт.


In [3]:
crawl_program = """
import json
import sys
from scrapy.crawler import CrawlerProcess
from scrapy.utils.project import get_project_settings
from earthquake_project.spiders.scraper import EarthquakesSpider

settings = get_project_settings()
settings.set("LOG_LEVEL", "ERROR")
settings.set("TELNETCONSOLE_ENABLED", False)
process = CrawlerProcess(settings)
crawler = process.create_crawler(EarthquakesSpider)
process.crawl(crawler, start=sys.argv[1], end=sys.argv[2], db_path=sys.argv[3], page_size="200")
process.start()
stats = crawler.stats.get_stats()
if stats.get("finish_reason") != "finished" or stats.get("log_count/ERROR", 0):
    raise RuntimeError("Сбор данных завершился с ошибками")
print(json.dumps({key: stats.get("events/" + key, 0)
                  for key in ["received", "inserted", "updated", "unchanged", "skipped"]}))
"""

def collect_period(start, end, label):
    completed = subprocess.run(
        [sys.executable, "-X", "utf8", "-c", crawl_program, start, end, str(report_db)],
        cwd=PROJECT_DIR, capture_output=True, text=True, encoding="utf-8", timeout=300,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr)
    stats = json.loads(completed.stdout.strip())
    with engine.connect() as connection:
        total = connection.scalar(select(func.count()).select_from(Earthquake))
    return {
        "Этап": label, "Получено": stats["received"], "Добавлено": stats["inserted"],
        "Обновлено": stats["updated"], "Без изменений": stats["unchanged"],
        "Пропущено": stats["skipped"], "Строк в базе": total,
    }


In [4]:
first_load = collect_period("2025-01-01", "2025-01-08", "1–7 января: первая загрузка")
assert first_load["Добавлено"] > 0
display(pd.DataFrame([first_load]))

with engine.connect() as connection:
    sample = pd.read_sql_query(
        select(Earthquake.event_id, Earthquake.occurred_at, Earthquake.magnitude,
               Earthquake.depth_km, Earthquake.place)
        .order_by(Earthquake.occurred_at, Earthquake.event_id).limit(5), connection,
    )
display(sample)


,Этап,Получено,Добавлено,Обновлено,Без изменений,Пропущено,Строк в базе
0,1–7 января: первая загрузка,585,585,0,0,0,585


,event_id,occurred_at,magnitude,depth_km,place
0,us6000pjr6,2025-01-01T00:13:22.650+00:00,4.2,35.000,"127 km WSW of Merizo Village, Guam"
1,us6000pgrt,2025-01-01T01:00:50.839+00:00,4.0,10.000,"228 km SE of Marsaskala, Malta"
2,us6000pgru,2025-01-01T01:07:10.492+00:00,4.4,277.355,"256 km SSE of Ambon, Indonesia"
3,us6000pmeg,2025-01-01T01:27:15.098+00:00,3.0,10.000,"206 km SSE of False Pass, Alaska"
4,us6000pjqz,2025-01-01T01:46:59.538+00:00,4.7,10.000,"67 km WNW of Rabaul, Papua New Guinea"


### 4.4. Повторная загрузка и догрузка

Повторно собирается первая неделя. Отсутствие новых строк при том же наборе событий подтверждает защиту от дубликатов. Затем собирается следующая неделя: количество строк увеличивается за счёт новых событий.


In [5]:
repeated_load = collect_period("2025-01-01", "2025-01-08", "1–7 января: повтор")
assert repeated_load["Получено"] > 0
assert repeated_load["Добавлено"] == 0
assert repeated_load["Строк в базе"] == first_load["Строк в базе"]
display(pd.DataFrame([repeated_load]))


,Этап,Получено,Добавлено,Обновлено,Без изменений,Пропущено,Строк в базе
0,1–7 января: повтор,585,0,0,585,0,585


In [6]:
second_load = collect_period("2025-01-08", "2025-01-15", "8–14 января: догрузка")
assert second_load["Добавлено"] > 0
assert second_load["Строк в базе"] == first_load["Строк в базе"] + second_load["Добавлено"]
load_results = pd.DataFrame([first_load, repeated_load, second_load])
display(load_results)


,Этап,Получено,Добавлено,Обновлено,Без изменений,Пропущено,Строк в базе
0,1–7 января: первая загрузка,585,585,0,0,0,585
1,1–7 января: повтор,585,0,0,585,0,585
2,8–14 января: догрузка,516,516,0,0,0,1101


### 4.5. Подготовка данных для агрегаций

pandas читает строки SQLite в DataFrame и выполняет группировки в памяти. SQLAlchemy строит запросы к SQLite, где и происходит агрегирование; pandas используется только для табличного отображения полученных ответов.

В обоих способах применяются одинаковые границы: магнитуды `<3`, `3–<4`, `4–<5`, `5–<6`, `≥6`; глубины `<70`, `70–<300`, `≥300` км. Неизвестная глубина не входит в группировку по глубине.


In [7]:
with engine.connect() as connection:
    events = pd.read_sql_query(select(Earthquake.__table__), connection)

events["day_utc"] = events["occurred_at"].str[:10]
events["hour_utc"] = events["occurred_at"].str[11:13]
events["magnitude_band"] = pd.cut(
    events["magnitude"], [-float("inf"), 3, 4, 5, 6, float("inf")],
    labels=["M < 3", "3 ≤ M < 4", "4 ≤ M < 5", "5 ≤ M < 6", "M ≥ 6"],
    right=False,
)
events["depth_band"] = pd.cut(
    events["depth_km"], [-float("inf"), 70, 300, float("inf")],
    labels=["мелкие (<70 км)", "промежуточные (70–300 км)", "глубокие (≥300 км)"],
    right=False,
)


print("Событий для анализа:", len(events))


Событий для анализа: 1101


In [8]:
e = Earthquake.__table__.c
day_sql = func.substr(e.occurred_at, 1, 10).label("day_utc")
hour_sql = func.substr(e.occurred_at, 12, 2).label("hour_utc")
magnitude_band_sql = case(
    (e.magnitude < 3, "M < 3"),
    (e.magnitude < 4, "3 ≤ M < 4"),
    (e.magnitude < 5, "4 ≤ M < 5"),
    (e.magnitude < 6, "5 ≤ M < 6"),
    else_="M ≥ 6",
).label("magnitude_band")
depth_band_sql = case(
    (e.depth_km < 70, "мелкие (<70 км)"),
    (e.depth_km < 300, "промежуточные (70–300 км)"),
    else_="глубокие (≥300 км)",
).label("depth_band")


### 4.6. Запрос 1. Число событий и магнитуда по дням

Определяется число событий, средняя и максимальная магнитуда для каждого дня UTC. Запрос позволяет сравнить активность в разные дни выбранного периода.


In [9]:
# pandas
pd_daily = (
    events.groupby("day_utc", observed=True)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         max_magnitude=("magnitude", "max"))
    .reset_index()
    .sort_values("day_utc")
)

display(Markdown("**pandas**"), pd_daily.round(3))

# SQLAlchemy
query = (
    select(day_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.max(e.magnitude).label("max_magnitude"))
    .group_by(day_sql)
    .order_by(day_sql)
)
with engine.connect() as connection:
    sql_daily = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_daily.round(3))


**pandas**

,day_utc,events,avg_magnitude,max_magnitude
0,2025-01-01,101,3.850,6.1
1,2025-01-02,72,4.146,6.1
2,2025-01-03,79,3.961,5.5
3,2025-01-04,84,4.012,5.7
4,2025-01-05,60,3.976,6.2
5,2025-01-06,82,4.035,5.2
6,2025-01-07,107,4.210,7.1
7,2025-01-08,88,4.112,5.5
8,2025-01-09,76,3.751,5.8
9,2025-01-10,83,3.863,5.6


**SQLAlchemy**

,day_utc,events,avg_magnitude,max_magnitude
0,2025-01-01,101,3.850,6.1
1,2025-01-02,72,4.146,6.1
2,2025-01-03,79,3.961,5.5
3,2025-01-04,84,4.012,5.7
4,2025-01-05,60,3.976,6.2
5,2025-01-06,82,4.035,5.2
6,2025-01-07,107,4.210,7.1
7,2025-01-08,88,4.112,5.5
8,2025-01-09,76,3.751,5.8
9,2025-01-10,83,3.863,5.6


### 4.7. Запрос 2. Распределение по диапазонам магнитуды

События группируются по диапазонам магнитуды. Для каждой группы вычисляются количество событий и средняя глубина.


In [10]:
# pandas
pd_magnitude = (
    events.groupby("magnitude_band", observed=True)
    .agg(events=("event_id", "size"), avg_depth_km=("depth_km", "mean"))
    .reset_index()
)

display(Markdown("**pandas**"), pd_magnitude.round(3))

# SQLAlchemy
query = (
    select(magnitude_band_sql, func.count().label("events"),
           func.avg(e.depth_km).label("avg_depth_km"))
    .group_by(magnitude_band_sql)
    .order_by(magnitude_band_sql)
)
with engine.connect() as connection:
    sql_magnitude = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_magnitude.round(3))


**pandas**

,magnitude_band,events,avg_depth_km
0,M < 3,225,27.362
1,3 ≤ M < 4,138,40.742
2,4 ≤ M < 5,659,62.531
3,5 ≤ M < 6,73,65.880
4,M ≥ 6,6,50.500


**SQLAlchemy**

,magnitude_band,events,avg_depth_km
0,3 ≤ M < 4,138,40.742
1,4 ≤ M < 5,659,62.531
2,5 ≤ M < 6,73,65.880
3,M < 3,225,27.362
4,M ≥ 6,6,50.500


### 4.8. Запрос 3. Распределение по глубине

События делятся на мелкие, промежуточные и глубокие. Рассчитываются количество и средняя магнитуда в каждой группе.


In [11]:
# pandas
pd_depth = (
    events.dropna(subset=["depth_km"])
    .groupby("depth_band", observed=True)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"))
    .reset_index()
)

display(Markdown("**pandas**"), pd_depth.round(3))

# SQLAlchemy
query = (
    select(depth_band_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"))
    .where(e.depth_km.is_not(None))
    .group_by(depth_band_sql)
    .order_by(depth_band_sql)
)
with engine.connect() as connection:
    sql_depth = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_depth.round(3))


**pandas**

,depth_band,events,avg_magnitude
0,мелкие (<70 км),891,3.937
1,промежуточные (70–300 км),169,4.089
2,глубокие (≥300 км),41,4.424


**SQLAlchemy**

,depth_band,events,avg_magnitude
0,глубокие (≥300 км),41,4.424
1,мелкие (<70 км),891,3.937
2,промежуточные (70–300 км),169,4.089


### 4.9. Запрос 4. Статистика по сетям каталога

Для каждой сети каталога определяются число записей, средняя и максимальная магнитуда. Сети сравниваются по вкладу в выборку.


In [12]:
# pandas
pd_network = (
    events.groupby("network", dropna=False)
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         max_magnitude=("magnitude", "max"))
    .reset_index()
    .sort_values(["events", "network"], ascending=[False, True])
)

display(Markdown("**pandas**"), pd_network.round(3))

# SQLAlchemy
query = (
    select(e.network, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.max(e.magnitude).label("max_magnitude"))
    .group_by(e.network)
    .order_by(func.count().desc(), e.network)
)
with engine.connect() as connection:
    sql_network = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_network.round(3))


**pandas**

,network,events,avg_magnitude,max_magnitude
9,us,894,4.225,7.10
0,ak,50,2.880,3.90
6,pr,46,3.117,3.90
3,nc,42,2.908,4.65
8,tx,19,2.758,4.20
2,hv,18,2.854,3.17
1,ci,16,2.791,3.10
5,nn,10,2.830,3.60
10,uu,4,2.820,3.10
4,nm,1,2.880,2.88


**SQLAlchemy**

,network,events,avg_magnitude,max_magnitude
0,us,894,4.225,7.10
1,ak,50,2.880,3.90
2,pr,46,3.117,3.90
3,nc,42,2.908,4.65
4,tx,19,2.758,4.20
5,hv,18,2.854,3.17
6,ci,16,2.791,3.10
7,nn,10,2.830,3.60
8,uu,4,2.820,3.10
9,nm,1,2.880,2.88


### 4.10. Запрос 5. Распределение по часам UTC

Для каждого часа UTC подсчитываются количество событий и средняя магнитуда. Все дни выборки рассматриваются совместно.


In [13]:
# pandas
pd_hour = (
    events.groupby("hour_utc")
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"))
    .reset_index()
    .sort_values("hour_utc")
)

display(Markdown("**pandas**"), pd_hour.round(3))

# SQLAlchemy
query = (
    select(hour_sql, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"))
    .group_by(hour_sql)
    .order_by(hour_sql)
)
with engine.connect() as connection:
    sql_hour = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_hour.round(3))


**pandas**

,hour_utc,events,avg_magnitude
0,00,49,4.118
1,01,67,4.177
2,02,50,4.264
3,03,51,4.089
4,04,63,3.916
5,05,39,3.856
6,06,41,3.944
7,07,47,3.914
8,08,35,4.121
9,09,40,3.980


**SQLAlchemy**

,hour_utc,events,avg_magnitude
0,00,49,4.118
1,01,67,4.177
2,02,50,4.264
3,03,51,4.089
4,04,63,3.916
5,05,39,3.856
6,06,41,3.944
7,07,47,3.914
8,08,35,4.121
9,09,40,3.981


### 4.11. Запрос 6. Сравнение по флагу tsunami

Сравниваются количество событий, средняя магнитуда и средняя значимость для значений флага tsunami 0 и 1.


In [14]:
# pandas
pd_tsunami = (
    events.groupby("tsunami")
    .agg(events=("event_id", "size"), avg_magnitude=("magnitude", "mean"),
         avg_significance=("significance", "mean"))
    .reset_index()
    .sort_values("tsunami")
)

display(Markdown("**pandas**"), pd_tsunami.round(3))

# SQLAlchemy
query = (
    select(e.tsunami, func.count().label("events"),
           func.avg(e.magnitude).label("avg_magnitude"),
           func.avg(e.significance).label("avg_significance"))
    .group_by(e.tsunami)
    .order_by(e.tsunami)
)
with engine.connect() as connection:
    sql_tsunami = pd.read_sql_query(query, connection)

display(Markdown("**SQLAlchemy**"), sql_tsunami.round(3))


**pandas**

,tsunami,events,avg_magnitude,avg_significance
0,0,1099,3.975,258.068
1,1,2,5.725,749.000


**SQLAlchemy**

,tsunami,events,avg_magnitude,avg_significance
0,0,1099,3.975,258.068
1,1,2,5.725,749.000


### 4.12. Сопоставление результатов

Проверяются состав групп, количество событий и вычисленные показатели. Результаты сравниваются до округления с допуском `1e-8` для чисел с плавающей точкой. Дополнительно сопоставляются расчёты в блокноте и в модуле `analysis.py`.


In [15]:
from analysis import pandas_queries, sqlalchemy_queries

pandas_results = {
    "daily": pd_daily, "magnitude": pd_magnitude, "depth": pd_depth,
    "network": pd_network, "hour": pd_hour, "tsunami": pd_tsunami,
}
sql_results = {
    "daily": sql_daily, "magnitude": sql_magnitude, "depth": sql_depth,
    "network": sql_network, "hour": sql_hour, "tsunami": sql_tsunami,
}
compare_results(pandas_results, sql_results)
compare_results(pandas_results, pandas_queries(engine))
compare_results(sql_results, sqlalchemy_queries(engine))
display(pd.DataFrame([
    {"Запрос": name, "Число групп": len(frame), "Сопоставление": "Совпадают"}
    for name, frame in pandas_results.items()
]))


,Запрос,Число групп,Сопоставление
0,daily,14,Совпадают
1,magnitude,5,Совпадают
2,depth,3,Совпадают
3,network,11,Совпадают
4,hour,24,Совпадают
5,tsunami,2,Совпадают


## 5. Результаты

Сводка загрузки и полученные агрегированные показатели приведены ниже. Числа вычислены по данным текущего выполнения блокнота.


In [16]:
display(load_results)
most_active = pd_daily.loc[pd_daily["events"].idxmax()]
largest_band = pd_magnitude.loc[pd_magnitude["events"].idxmax()]
largest_network = pd_network.loc[pd_network["events"].idxmax()]
busiest_hour = pd_hour.loc[pd_hour["events"].idxmax()]
display(Markdown(
    f"- В SQLite сохранено **{len(events)}** событий. Повтор первой недели добавил "
    f"**{repeated_load['Добавлено']}** строк, догрузка второй недели — **{second_load['Добавлено']}**.\n"
    f"- Наибольшее число событий за день: **{int(most_active['events'])}**, "
    f"**{most_active['day_utc']}**. Максимальная магнитуда в этот день — **{most_active['max_magnitude']:.1f}**.\n"
    f"- Самая многочисленная группа магнитуды — **{largest_band['magnitude_band']}**: "
    f"**{int(largest_band['events'])}** событий.\n"
    f"- Больше всего записей предоставила сеть **{largest_network['network']}**: "
    f"**{int(largest_network['events'])}**.\n"
    f"- Наиболее частый час UTC — **{busiest_hour['hour_utc']}:00–{busiest_hour['hour_utc']}:59**: "
    f"**{int(busiest_hour['events'])}** событий.\n"
    "- Все **6** пар агрегаций pandas и SQLAlchemy совпали."
))
display(pd_depth.round(3), pd_tsunami.round(3))
print("Файл базы опыта:", report_db.relative_to(PROJECT_DIR).as_posix())
engine.dispose()


,Этап,Получено,Добавлено,Обновлено,Без изменений,Пропущено,Строк в базе
0,1–7 января: первая загрузка,585,585,0,0,0,585
1,1–7 января: повтор,585,0,0,585,0,585
2,8–14 января: догрузка,516,516,0,0,0,1101


- В SQLite сохранено **1101** событий. Повтор первой недели добавил **0** строк, догрузка второй недели — **516**.
- Наибольшее число событий за день: **107**, **2025-01-07**. Максимальная магнитуда в этот день — **7.1**.
- Самая многочисленная группа магнитуды — **4 ≤ M < 5**: **659** событий.
- Больше всего записей предоставила сеть **us**: **894**.
- Наиболее частый час UTC — **01:00–01:59**: **67** событий.
- Все **6** пар агрегаций pandas и SQLAlchemy совпали.

,depth_band,events,avg_magnitude
0,мелкие (<70 км),891,3.937
1,промежуточные (70–300 км),169,4.089
2,глубокие (≥300 км),41,4.424


,tsunami,events,avg_magnitude,avg_significance
0,0,1099,3.975,258.068
1,1,2,5.725,749.000


Файл базы опыта: data/report_runs/report_20261005_184350_931422.sqlite


## 6. Выводы по проделанной работе

Данные о землетрясениях получены непосредственно с веб-ресурса USGS через API средствами Scrapy. События преобразованы из GeoJSON и сохранены в SQLite. Первичный ключ и проверка времени обновления позволяют добавлять новые данные без дубликатов и принимать уточнения существующих записей.

Выполнены шесть разных запросов для агрегации: по дням, диапазонам магнитуды, диапазонам глубины, сетям каталога, часам UTC и флагу tsunami. Каждый запрос реализован с использованием pandas и SQLAlchemy. Сопоставление результатов подтвердило согласованность двух способов обработки.

Эксперимент продемонстрировал сохранение первой выборки, повторную загрузку без увеличения числа строк и догрузку следующего периода. Таким образом, реализованы все этапы поставленной задачи: сбор, хранение, пополнение базы и агрегирование данных.

Полученные показатели описывают только выбранный двухнедельный период и события магнитудой от 2,5. Распределение по часам не доказывает суточную закономерность землетрясений. Код сети не является географическим регионом, а флаг tsunami не подтверждает факт возникновения цунами. Значения каталога могут уточняться после выполнения работы.
